[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Python from the Start](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)

# Modules and Imports


## What you will be able to do

Move code out of a notebook into a `.py` file and import it back, choose between the import
forms, and recognize the reason an edited module keeps giving you the old answer.


## The idea

### The problem

The **Functions** notebook solved the problem of running the same code twice. It did not solve
running it in a different notebook, or next week, or from a program that is not a notebook at
all.

Right now every function you have written lives in the cell that defined it. Open a new notebook
and it is gone. Copy the cell across and you have two versions that will drift apart, which is
the same problem functions were introduced to fix, one level up.

There is also a limit that arrives sooner than you expect. A notebook of two hundred cells is
hard to navigate and impossible to test, and the useful parts are buried among the experiments.

### What a module is

> A **module** is a `.py` file. Its contents become available to other code through `import`,
> which runs the file once and gives you access to the names it defined.
>
> A **package** is a folder of modules. `import` treats it much the same way, with dots for the
> folders: `from pathlib import Path`.

You have been importing since the **Numbers** notebook. `math`, `re`, `pathlib` and `shutil` are
all modules that ship with Python. The only new idea here is writing one yourself.

### The forms of import, and which to use

There are four, and they differ in what ends up in your namespace:

| Written | You then use | Good for |
|---|---|---|
| `import math` | `math.sqrt(2)` | the default; the source stays visible |
| `import numpy as np` | `np.array(...)` | long names, where the short form is conventional |
| `from math import sqrt` | `sqrt(2)` | one or two names used constantly |
| `from math import *` | `sqrt(2)` | nothing; avoid it |

The last one imports every public name at once, which means you no longer know where anything
came from, and a later import can silently replace a name you were using. Every style guide
tells you not to, and the **Scope** notebook showed what a quietly replaced name costs.

### The part that will catch you in a notebook

Python imports a module **once per session**. Import it again and you get the copy already in
memory, without the file being read.

That is a sensible optimization for a program that starts, runs and exits. In a notebook, where
you edit a module and re-run the import expecting your change to take effect, it means you keep
getting the old version with no indication that anything is stale. It is confusing enough to be
worth knowing before it happens rather than after.

### Where you will meet this

Every library you use is modules. The **Testing and Packaging** guide is about turning a folder
of them into something installable. The **Object-Oriented Python** and **APIs and JSON** guides
both assume you can split code across files.

### A note on what this notebook creates

Setup makes a folder called `scratch` next to this notebook and writes a small module into it.
The last cell deletes the folder. Nothing outside it is touched.

### What this notebook covers

- Writing a `.py` file and importing it
- The four import forms, and why one of them is a mistake
- What a module carries: its name, its docstring, and the names it defines
- `sys.path`, which is where Python looks
- The import cache, and `importlib.reload`
- `if __name__ == "__main__"`, and what it is actually for
- Three errors, including the one that looks like your edit did nothing

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet:
read it, and read the output underneath it. Everything from Setup onward is where you start
running things, and the rest of the notebook takes this apart piece by piece.

```python
# readings.py, saved as a file
import math

def mean(values):
    return sum(values) / len(values)

def deviation(values):
    m = mean(values)
    return math.sqrt(sum((v - m) ** 2 for v in values) / len(values))
```

```python
# any other notebook or program
import readings

temperatures = [18, 21, 19, 24, 22]
print(readings.mean(temperatures))
print(round(readings.deviation(temperatures), 2))
```

```
20.8
2.14
```

One file, written once, usable from anywhere that can find it. Note that `readings.py` imports
`math` for itself; a module is ordinary Python and can import whatever it needs.


## Setup

Three imports, a folder to work in, and a small module written to disk. `sys.path` is extended
so Python can find it.

**Run this cell before the rest of the notebook.**


In [1]:
import sys
import shutil
import importlib
from pathlib import Path

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)

# The module is written as one string. ''' quotes it so the """ docstrings inside survive.
module_code = '''
"""Summarize a list of sensor readings."""

import math

UNIT = "degrees"


def mean(values):
    """Return the arithmetic mean."""
    return sum(values) / len(values)


def spread(values):
    """Return the difference between the largest and smallest value."""
    return max(values) - min(values)


def deviation(values):
    """Return the standard deviation."""
    m = mean(values)
    return math.sqrt(sum((v - m) ** 2 for v in values) / len(values))


def summary(values):
    """Return a one-line description of the readings."""
    return (f"n={len(values)}  mean={mean(values):.2f} {UNIT}  "
            f"spread={spread(values)}  sd={deviation(values):.2f}")


if __name__ == "__main__":
    print("run directly:", summary([18, 21, 19, 24, 22]))
'''

(scratch / "readings.py").write_text(module_code)

if str(scratch.resolve()) not in sys.path:
    sys.path.insert(0, str(scratch.resolve()))

print("wrote", scratch / "readings.py")


wrote scratch/readings.py


## Worked examples

### Importing your own module

The file exists, and Python can find it. `import` runs it once and hands back a module object.


In [2]:
import readings

temperatures = [18, 21, 19, 24, 22]

print(readings.mean(temperatures))
print(readings.spread(temperatures))
print(readings.summary(temperatures))


20.8
6
n=5  mean=20.80 degrees  spread=6  sd=2.14


The name before the dot is the module, and the name after it is something defined inside. That
is the same dot the **Values and Variables** notebook introduced for reaching what a value
carries.

### The other import forms


In [3]:
import readings as r

print(r.summary(temperatures))


n=5  mean=20.80 degrees  spread=6  sd=2.14


In [4]:
from readings import mean, spread

print(mean(temperatures))
print(spread(temperatures))


20.8
6


`from ... import` puts the names directly in your namespace, so you write `mean` rather than
`readings.mean`. That reads well when a name is used many times, and it costs you the
information about where it came from.

The cost is real. Six months later, reading a file with fifty imported names and no prefixes, no
one can tell which library `parse` belongs to. `import x` keeps that visible, which is why it is
the default.

### What a module carries


In [5]:
print("name:  ", readings.__name__)
print("doc:   ", readings.__doc__)
print("public:", [n for n in dir(readings) if not n.startswith("_")])


name:   readings
doc:    Summarize a list of sensor readings.
public: ['UNIT', 'deviation', 'math', 'mean', 'spread', 'summary']


`dir()` lists everything the module defines. The names beginning with an underscore are
Python's own bookkeeping, which is why they are filtered out here.

`math` is in that list, which surprises people. `readings.py` imports `math` for its own use,
and an import creates a name in the module doing the importing. So `readings.math` works, and
is the same `math` you would get yourself. It is not something to rely on, but it explains why
a module's public names include things it never defined.

`help(readings.deviation)` prints the docstring, which is the same `help` from the
**Running Python** notebook now working on a file you wrote.


In [6]:
help(readings.deviation)


Help on function deviation in module readings:

deviation(values)
    Return the standard deviation.



### Where Python looks

`import x` searches a list of directories, in order, and uses the first `x.py` it finds.


In [7]:
print("number of places Python will look:", len(sys.path))
print("is our scratch folder among them?", str(scratch.resolve()) in sys.path)


number of places Python will look: 7
is our scratch folder among them? True


That list is `sys.path`. It starts with the directory of the running script or notebook, then
the standard library, then anything installed with `pip`.

Setup added the scratch folder to the front, which is why `import readings` worked at all. In
normal use you would not touch `sys.path`: you put your module next to the code that imports it,
or install it properly, which the **Testing and Packaging** guide covers.

The order matters. A file called `random.py` in your own folder is found **before** the standard
library's `random`, and everything that expected the real one breaks. Avoid naming your files
after modules you use.


### The import cache

This is the one that wastes an afternoon in a notebook.


In [8]:
edited = 'UNIT = "kelvin"\n\ndef mean(values):\n    return sum(values) / len(values)\n'
(scratch / "readings.py").write_text(edited)

print("the file on disk now says:", (scratch / "readings.py").read_text().splitlines()[0])


the file on disk now says: UNIT = "kelvin"


In [9]:
import readings

print("but importing it gives:", readings.UNIT)


but importing it gives: degrees


The file changed and the import did not notice. Python keeps every imported module in memory for
the length of the session, and a second `import` hands back what is already there without
reading the file.

No error, no warning. In a notebook, where the runtime lives for hours, an edit to a module can
appear to do nothing at all.

`importlib.reload` forces the file to be read again.


In [10]:
importlib.reload(readings)

print("after reload:", readings.UNIT)


after reload: kelvin


Two things to know about `reload`. It only affects the module object itself, so names already
pulled out with `from readings import mean` still point at the old function. And it is a
development convenience rather than something to rely on in finished code.

The dependable fix is **Runtime > Restart session** in Colab, which clears everything. Reach for
`reload` while iterating, and restart when something does not add up.


### if __name__ == "__main__"

Every module has a `__name__`. When you import it, `__name__` is the module's name. When you run
the file directly, `__name__` is the string `"__main__"`.


In [11]:
importlib.reload(readings)

print("imported, so __name__ is:", readings.__name__)


imported, so __name__ is: readings


That difference lets one file be both a library and a program. Code under
`if __name__ == "__main__":` runs only when the file is executed directly, and is skipped when
the file is imported.

The module Setup wrote has exactly that guard. Running the file as a program shows it:


In [12]:
import subprocess

demo_code = '''
def convert(celsius):
    return celsius * 9 / 5 + 32

print("this line runs on import AND when run directly")

if __name__ == "__main__":
    print("this line runs only when run directly:", convert(100))
'''

(scratch / "demo.py").write_text(demo_code)

result = subprocess.run([sys.executable, str(scratch / "demo.py")],
                        capture_output=True, text=True)
print(result.stdout.rstrip())


this line runs on import AND when run directly
this line runs only when run directly: 212.0


In [13]:
import demo

print("--- and now imported, which prints only the first line ---")


this line runs on import AND when run directly
--- and now imported, which prints only the first line ---


Both lines appeared when the file was run. Only the unguarded one appeared on import.

Use the guard for anything a file should do as a program and not as a library: a quick test, a
command line entry point, an example. Without it, importing the module would run all of that
as a side effect.


### Cleaning up


In [14]:
shutil.rmtree(scratch)

print("scratch still there:", scratch.exists())


scratch still there: False


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Start by re-creating a folder to work in, since the cleanup above removed it:

```
from pathlib import Path
import sys
work = Path("practice")
work.mkdir(exist_ok=True)
sys.path.insert(0, str(work.resolve()))
```

Try each one before you look at an answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/18-modules-and-imports-solutions.ipynb).

**1.** Create the folder and path setup above, then write `practice/shapes.py` containing a
function `area(width, height)` that returns their product.


In [15]:
# your code here


**2.** Import the module and call `area(3, 4)`, using the `import shapes` form.


In [16]:
# your code here


**3.** Import just the function with `from shapes import area` and call it again.


In [17]:
# your code here


**4.** Print the module's `__name__` and the list of its public names.


In [18]:
# your code here


**5.** Edit `shapes.py` to add a `perimeter(width, height)` function, import the module again,
and try to call `shapes.perimeter(3, 4)`. Predict what happens before you run it.


In [19]:
# your code here


**6.** Make the new function available with `importlib.reload`, call it, then delete the
`practice` folder.


In [20]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### ModuleNotFoundError: Python cannot find it


In [21]:
import notarealmodule


ModuleNotFoundError: No module named 'notarealmodule'

`No module named 'notarealmodule'` means the name was not found anywhere in `sys.path`. Three
causes, in order of likelihood: a typo, a file in a different folder from the one you are
running in, or a package that was never installed. The **Environments and pip** notebook covers
the third.


### ImportError: the module is there, the name is not


In [22]:
try:
    from math import square_root
except ImportError as e:
    print(type(e).__name__)
    print(str(e).split(" (")[0])


ImportError
cannot import name 'square_root' from 'math'


`cannot import name 'square_root' from 'math'` is a different message from the one above, and
the difference is the useful part: `math` was found and read, and it has no `square_root`. It
is called `sqrt`.

The cell above catches the error so the output stays short. Uncaught, the message also names
the file the module was loaded from, which is useful when two modules share a name and you need
to know which one you actually got.

`dir(math)` lists what a module offers when you are not sure.


In [23]:
import math

print([n for n in dir(math) if n.startswith("s")])


['sin', 'sinh', 'sqrt', 'sumprod']


### The quiet one: the edit that did nothing

Covered above, and worth stating as an error because it produces no error.

A module edited after it was imported keeps giving the old answers for the rest of the session.
There is no message, and the code looks correct because it is correct, just not the version
running.

If a change to a `.py` file appears to have no effect, that is the first thing to suspect.
`importlib.reload(module)` re-reads it, and restarting the runtime is the version that always
works.


## Recap

- A **module** is a `.py` file; `import` runs it once and gives you the names it defined.
- `import x` keeps the source of every name visible, and is the default for that reason.
- `from x import y` is fine for names used constantly; `from x import *` hides where things came
  from and should not be used.
- `dir(module)` lists what it offers and `help()` reads its docstrings.
- `sys.path` is the search order, and a local file named after a standard module shadows it.
- Python caches imports for the session, so editing a module changes nothing until you
  `importlib.reload` it or restart the runtime.
- `if __name__ == "__main__":` runs only when the file is executed directly, never on import.


## What is next

The **Environments and pip** notebook, which covers the code you did not write: installing
packages, why the same program works on one machine and not another, and what a virtual
environment is for.


---

&#8592; **Previous:** [Files and Paths](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/17-files-and-paths.ipynb)  &nbsp;·&nbsp;  [Python from the Start Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)
